In [1]:
import numpy as np
import pandas as pd


# What You Are Answering

<br>Which campaign drove the highest revenue uplift?</br>
<br>What was the ROI of each promotional campaign?</br>
<br>Did promotions drive real volume increase or just give away margin?</br>

In [2]:
sales_df=pd.read_excel("Sales Data.xlsx",sheet_name='Sales_Data')

In [3]:
sales_df.shape

(864, 17)

In [4]:
sales_df.dtypes

ID                            int64
Month                        object
Month Year           datetime64[ns]
Market                       object
Channel                      object
SKU                          object
SKU-MARKET                   object
Units_Sold                    int64
Selling_Price_USD           float64
Revenue USD                 float64
COGS_USD                      int64
Gross_Margin_USD            float64
Gross_Margin_%                int64
Promotion_Flag               object
Promo_Discount_%              int64
Campaign_Name                object
Data_Type                    object
dtype: object

In [5]:
sales_df.head(5)

,ID,Month,Month Year,Market,Channel,SKU,SKU-MARKET,Units_Sold,Selling_Price_USD,Revenue USD,COGS_USD,Gross_Margin_USD,Gross_Margin_%,Promotion_Flag,Promo_Discount_%,Campaign_Name,Data_Type
0,1,January,2025-01-01,USA,Sephora,Vanilla 28,Vanilla 28-USA,19250,105.0,2021250.0,606375,1414875.0,70,No,0,NaN,Stimulated
1,2,January,2025-01-01,USA,Sephora,Eden Juicy Apple 01,Eden Juicy Apple 01-USA,11000,105.0,1155000.0,358050,796950.0,69,No,0,NaN,Stimulated
2,3,January,2025-01-01,USA,Sephora,Oudgasm Tobacco Oud 04,Oudgasm Tobacco Oud 04-USA,7425,140.0,1039500.0,363825,675675.0,65,No,0,NaN,Stimulated
3,4,January,2025-01-01,USA,Sephora,Yum Pistachio Gelato 33,Yum Pistachio Gelato 33-USA,6600,105.0,693000.0,214830,478170.0,69,No,0,NaN,Stimulated
4,5,January,2025-01-01,USA,Sephora,Freedom Musk Santal 34,Freedom Musk Santal 34-USA,5500,105.0,577500.0,184800,392700.0,68,No,0,NaN,Stimulated


In [6]:
sales_df['Promotion_Flag'].value_counts()

Promotion_Flag
No     576
Yes    288
Name: count, dtype: int64

In [7]:
sales_df['Campaign_Name'].value_counts()

Campaign_Name
Valentine's Day    72
Freedom Launch     72
Summer Sale        48
Black Friday       48
Eid Al Fitr        24
Eid Al Adha        24
Name: count, dtype: int64

In [8]:
promo_df=sales_df[sales_df['Promotion_Flag']=='Yes']

In [9]:
non_promo_df=sales_df[sales_df['Promotion_Flag']=='No']

In [10]:
print("Promo rows:", promo_df.shape[0])
print("Non-promo rows:", non_promo_df.shape[0])
print("Total:", promo_df.shape[0] + non_promo_df.shape[0])

Promo rows: 288
Non-promo rows: 576
Total: 864


In [11]:
non_promo_df['Campaign_Name']

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
859    NaN
860    NaN
861    NaN
862    NaN
863    NaN
Name: Campaign_Name, Length: 576, dtype: object

Calculate Non-Promo Baseline Revenue per SKU per Market

In [12]:
non_promo_df.columns

Index(['ID', 'Month', 'Month Year', 'Market', 'Channel', 'SKU', 'SKU-MARKET',
       'Units_Sold', 'Selling_Price_USD', 'Revenue USD', 'COGS_USD',
       'Gross_Margin_USD', 'Gross_Margin_%', 'Promotion_Flag',
       'Promo_Discount_%', 'Campaign_Name', 'Data_Type'],
      dtype='object')

In [13]:
baseline_monthly_df=pd.DataFrame(non_promo_df.groupby(['SKU','Market']).agg(Total_NonPromo_Revenue=('Revenue USD', 'sum'),
    Avg_Monthly_Revenue_Baseline=('Revenue USD', 'mean'),
    NonPromo_Month_Count=('Month Year', 'count'),
    Total_NonPromo_Units=('Units_Sold','sum'),
    Avg_Monthly_Units_Baseline=('Units_Sold','mean')).reset_index())

In [14]:
baseline_monthly_df

,SKU,Market,Total_NonPromo_Revenue,Avg_Monthly_Revenue_Baseline,NonPromo_Month_Count,Total_NonPromo_Units,Avg_Monthly_Units_Baseline
0,Eden Juicy Apple 01,UAE,3150000.0,98437.50,32,27777,868.03125
1,Eden Juicy Apple 01,UK,3498000.0,109312.50,32,34431,1075.96875
2,Eden Juicy Apple 01,USA,22260000.0,695625.00,32,212000,6625.00000
3,Freedom Musk Santal 34,UAE,2100000.0,65625.00,32,18516,578.62500
4,Freedom Musk Santal 34,UK,1590000.0,49687.50,32,15650,489.06250
5,Freedom Musk Santal 34,USA,11130000.0,347812.50,32,106000,3312.50000
6,Musk 12,UAE,1680000.0,52500.00,32,14816,463.00000
7,Musk 12,UK,1113000.0,34781.25,32,10954,342.31250
8,Musk 12,USA,5565000.0,173906.25,32,53000,1656.25000
9,Oudgasm Tobacco Oud 04,UAE,7350000.0,229687.50,32,49405,1543.90625


In [14]:
promo_df.columns


Index(['ID', 'Month', 'Month Year', 'Market', 'Channel', 'SKU', 'SKU-MARKET',
       'Units_Sold', 'Selling_Price_USD', 'Revenue USD', 'COGS_USD',
       'Gross_Margin_USD', 'Gross_Margin_%', 'Promotion_Flag',
       'Promo_Discount_%', 'Campaign_Name', 'Data_Type'],
      dtype='object')

In [15]:
promo_grouped_df=pd.DataFrame(promo_df.groupby(['Campaign_Name','SKU','Market']).agg(Promo_Revenue_USD=('Revenue USD','sum'),
                                                       Promo_Units_Sold=('Units_Sold','sum'),
                                                       Avg_of_Promo_Discount_percent=('Promo_Discount_%','mean'),
                                                       Promo_Month_Count=('Month Year','count')).reset_index())

In [16]:
promo_grouped_df

,Campaign_Name,SKU,Market,Promo_Revenue_USD,Promo_Units_Sold,Avg_of_Promo_Discount_percent,Promo_Month_Count
0,Black Friday,Eden Juicy Apple 01,UK,616000.0,6063,25.0,4
1,Black Friday,Eden Juicy Apple 01,USA,3920000.0,37333,25.0,4
2,Black Friday,Freedom Musk Santal 34,UK,280000.0,2755,25.0,4
3,Black Friday,Freedom Musk Santal 34,USA,1960000.0,18667,25.0,4
4,Black Friday,Musk 12,UK,196000.0,1929,25.0,4
...,...,...,...,...,...,...,...
67,Valentine's Day,Vanilla 28,UK,720000.0,7087,15.0,4
68,Valentine's Day,Vanilla 28,USA,5880000.0,56000,15.0,4
69,Valentine's Day,Yum Pistachio Gelato 33,UAE,360000.0,3175,15.0,4
70,Valentine's Day,Yum Pistachio Gelato 33,UK,312000.0,3071,15.0,4


In [16]:
promo_grouped_df.columns

Index(['Campaign_Name', 'SKU', 'Market', 'Promo_Revenue_USD',
       'Promo_Units_Sold', 'Avg_of_Promo_Discount_percent',
       'Promo_Month_Count'],
      dtype='object')

In [17]:
baseline_monthly_df.columns

Index(['SKU', 'Market', 'Total_NonPromo_Revenue',
       'Avg_Monthly_Revenue_Baseline', 'NonPromo_Month_Count',
       'Total_NonPromo_Units', 'Avg_Monthly_Units_Baseline'],
      dtype='object')

In [18]:
promo_analysis_df=promo_grouped_df.merge(baseline_monthly_df,on=['SKU','Market'])

In [19]:
promo_analysis_df

,Campaign_Name,SKU,Market,Promo_Revenue_USD,Promo_Units_Sold,Avg_of_Promo_Discount_percent,Promo_Month_Count,Total_NonPromo_Revenue,Avg_Monthly_Revenue_Baseline,NonPromo_Month_Count,Total_NonPromo_Units,Avg_Monthly_Units_Baseline
0,Black Friday,Eden Juicy Apple 01,UK,616000.0,6063,25.0,4,3498000.0,109312.5,32,34431,1075.96875
1,Freedom Launch,Eden Juicy Apple 01,UK,440000.0,4331,0.0,4,3498000.0,109312.5,32,34431,1075.96875
2,Summer Sale,Eden Juicy Apple 01,UK,374000.0,3680,20.0,4,3498000.0,109312.5,32,34431,1075.96875
3,Valentine's Day,Eden Juicy Apple 01,UK,528000.0,5197,15.0,4,3498000.0,109312.5,32,34431,1075.96875
4,Black Friday,Eden Juicy Apple 01,USA,3920000.0,37333,25.0,4,22260000.0,695625.0,32,212000,6625.00000
...,...,...,...,...,...,...,...,...,...,...,...,...
67,Valentine's Day,Vanilla 28,UAE,600000.0,5291,15.0,4,4200000.0,131250.0,32,37042,1157.56250
68,Eid Al Adha,Yum Pistachio Gelato 33,UAE,270000.0,2380,15.0,4,2520000.0,78750.0,32,22223,694.46875
69,Eid Al Fitr,Yum Pistachio Gelato 33,UAE,270000.0,2380,20.0,4,2520000.0,78750.0,32,22223,694.46875
70,Freedom Launch,Yum Pistachio Gelato 33,UAE,300000.0,2645,0.0,4,2520000.0,78750.0,32,22223,694.46875


What is Uplift?Uplift simply means the extra something you gained because of the promotion.Example: 
A shop normally sells 100 bottles of perfume per month. During a Valentine's Day promotion they sell 140 bottles. The uplift is 40 bottles — the extra sales that happened because of the promotion.
Without the promotion those 40 extra sales would not have happened. That is the uplift — the incremental gain driven directly by the campaign.

In [19]:
promo_analysis_df['Promo_Month_Count'].unique()

array([4], dtype=int64)

## Calculate Uplift Metrics
## Calculate Expected Baseline Revenue During Promo Period

In [20]:
promo_analysis_df.columns

Index(['Campaign_Name', 'SKU', 'Market', 'Promo_Revenue_USD',
       'Promo_Units_Sold', 'Avg_of_Promo_Discount_percent',
       'Promo_Month_Count', 'Total_NonPromo_Revenue',
       'Avg_Monthly_Revenue_Baseline', 'NonPromo_Month_Count',
       'Total_NonPromo_Units', 'Avg_Monthly_Units_Baseline'],
      dtype='object')

In [21]:
promo_analysis_df.head(1)

,Campaign_Name,SKU,Market,Promo_Revenue_USD,Promo_Units_Sold,Avg_of_Promo_Discount_percent,Promo_Month_Count,Total_NonPromo_Revenue,Avg_Monthly_Revenue_Baseline,NonPromo_Month_Count,Total_NonPromo_Units,Avg_Monthly_Units_Baseline
0,Black Friday,Eden Juicy Apple 01,UK,616000.0,6063,25.0,4,3498000.0,109312.5,32,34431,1075.96875


In [22]:
promo_analysis_df['Expected_Baseline_Revenue']=promo_analysis_df['Avg_Monthly_Revenue_Baseline'] * promo_analysis_df['Promo_Month_Count']

In [23]:
promo_analysis_df['Expected_Baseline_Units']=promo_analysis_df['Avg_Monthly_Units_Baseline'] * promo_analysis_df['Promo_Month_Count']

Calculate Revenue Uplift

In [24]:
promo_analysis_df['Revenue_Uplift']=promo_analysis_df['Promo_Revenue_USD']-promo_analysis_df['Expected_Baseline_Revenue']

Calculate Discount Cost

In [25]:
promo_analysis_df['Discount_Cost']=promo_analysis_df['Expected_Baseline_Revenue']*(promo_analysis_df['Avg_of_Promo_Discount_percent'] / 100)

Calculate Promotional ROI

In [26]:
#Promo ROI answers this specific business question:

#"For every $1 Kayali spent on discounts — how many dollars of extra revenue did they get back?"
#ROI = 150% → promotion generated $1.50 of uplift for every $1.00 of discount given — profitable promotion
#ROI = 80% → promotion generated only $0.80 of uplift for every $1.00 of discount — promotion cost more than it generated
#ROI = 0% → promotion drove zero uplift — customers would have bought anyway at full price

ROI = ( WHAT YOU GOT/ WHAT YOU SPENT)*100

In [27]:
promo_analysis_df['Promo_ROI']=(promo_analysis_df['Revenue_Uplift'] / promo_analysis_df['Discount_Cost'] ) * 100


Calculate Volume Uplift %

"Are we giving discounts to loyal customers who would have bought anyway — or are we genuinely attracting new buyers with our promotions?"

In [28]:
promo_analysis_df['Volume_Uplift_percent']=((promo_analysis_df['Promo_Units_Sold']-promo_analysis_df['Expected_Baseline_Units']) / promo_analysis_df['Expected_Baseline_Units'])* 100 

Aggregate to Campaign Level

In [29]:
promo_analysis_df.columns

Index(['Campaign_Name', 'SKU', 'Market', 'Promo_Revenue_USD',
       'Promo_Units_Sold', 'Avg_of_Promo_Discount_percent',
       'Promo_Month_Count', 'Total_NonPromo_Revenue',
       'Avg_Monthly_Revenue_Baseline', 'NonPromo_Month_Count',
       'Total_NonPromo_Units', 'Avg_Monthly_Units_Baseline',
       'Expected_Baseline_Revenue', 'Expected_Baseline_Units',
       'Revenue_Uplift', 'Discount_Cost', 'Promo_ROI',
       'Volume_Uplift_percent'],
      dtype='object')

In [30]:
campaign_summary_df=promo_analysis_df.groupby(['Campaign_Name'])[['Promo_Revenue_USD','Expected_Baseline_Revenue','Revenue_Uplift','Discount_Cost','Promo_Units_Sold','Expected_Baseline_Units']].sum()

# Why Averaging ROI is Wrong
## Averaging ROI gives equal weight to every row regardless of size. Musk 12 with a tiny $30K uplift influences the average just as much as Vanilla 28 with $120K uplift. That is mathematically incorrect.
## Recalculating from totals gives the correct weighted picture — larger SKUs naturally have more influence on the campaign ROI because their numbers are bigger.

In [32]:
#we are recalculating PROMO ROI, Volume uplift % net promo gain for each campaign ad not for SKU's as it is there in the promo analysis df

In [33]:
campaign_summary_df

,Promo_Revenue_USD,Expected_Baseline_Revenue,Revenue_Uplift,Discount_Cost,Promo_Units_Sold,Expected_Baseline_Units
Campaign_Name,,,,,,
Black Friday,22400000.0,15900000.0,6500000.0,3975000.0,204199,144946.375
Eid Al Adha,2250000.0,2625000.0,-375000.0,393750.0,18189,21222.375
Eid Al Fitr,2250000.0,2625000.0,-375000.0,525000.0,18189,21222.375
Freedom Launch,18500000.0,18525000.0,-25000.0,0.0,166068,166168.750
Summer Sale,13600000.0,15900000.0,-2300000.0,3180000.0,123979,144946.375
Valentine's Day,22200000.0,18525000.0,3675000.0,2778750.0,199283,166168.750


In [34]:

campaign_summary_df['Campaign_Promo_ROI'] = (campaign_summary_df['Revenue_Uplift'] / campaign_summary_df['Discount_Cost']) * 100

In [35]:
campaign_summary_df['Campaign_Volume_Uplift_percent']=((campaign_summary_df['Promo_Units_Sold']-campaign_summary_df['Expected_Baseline_Units'])/campaign_summary_df['Expected_Baseline_Units'])*100

In [36]:
campaign_summary_df['Campaign_Net_Promo_Gain']=campaign_summary_df['Revenue_Uplift']-campaign_summary_df['Discount_Cost']

In [37]:
campaign_summary_df

,Promo_Revenue_USD,Expected_Baseline_Revenue,Revenue_Uplift,Discount_Cost,Promo_Units_Sold,Expected_Baseline_Units,Campaign_Promo_ROI,Campaign_Volume_Uplift_percent,Campaign_Net_Promo_Gain
Campaign_Name,,,,,,,,,
Black Friday,22400000.0,15900000.0,6500000.0,3975000.0,204199,144946.375,163.522013,40.878997,2525000.0
Eid Al Adha,2250000.0,2625000.0,-375000.0,393750.0,18189,21222.375,-95.238095,-14.293287,-768750.0
Eid Al Fitr,2250000.0,2625000.0,-375000.0,525000.0,18189,21222.375,-71.428571,-14.293287,-900000.0
Freedom Launch,18500000.0,18525000.0,-25000.0,0.0,166068,166168.750,-inf,-0.060631,-25000.0
Summer Sale,13600000.0,15900000.0,-2300000.0,3180000.0,123979,144946.375,-72.327044,-14.465608,-5480000.0
Valentine's Day,22200000.0,18525000.0,3675000.0,2778750.0,199283,166168.750,132.253711,19.928085,896250.0


In [38]:
campaign_summary_df['Campaign_Promo_ROI'] = campaign_summary_df['Campaign_Promo_ROI'].replace(
    [-float('inf'), float('inf')], None
)

In [39]:
campaign_summary_df.sort_values('Campaign_Promo_ROI', ascending=False)

,Promo_Revenue_USD,Expected_Baseline_Revenue,Revenue_Uplift,Discount_Cost,Promo_Units_Sold,Expected_Baseline_Units,Campaign_Promo_ROI,Campaign_Volume_Uplift_percent,Campaign_Net_Promo_Gain
Campaign_Name,,,,,,,,,
Black Friday,22400000.0,15900000.0,6500000.0,3975000.0,204199,144946.375,163.522013,40.878997,2525000.0
Valentine's Day,22200000.0,18525000.0,3675000.0,2778750.0,199283,166168.750,132.253711,19.928085,896250.0
Eid Al Fitr,2250000.0,2625000.0,-375000.0,525000.0,18189,21222.375,-71.428571,-14.293287,-900000.0
Summer Sale,13600000.0,15900000.0,-2300000.0,3180000.0,123979,144946.375,-72.327044,-14.465608,-5480000.0
Eid Al Adha,2250000.0,2625000.0,-375000.0,393750.0,18189,21222.375,-95.238095,-14.293287,-768750.0
Freedom Launch,18500000.0,18525000.0,-25000.0,0.0,166068,166168.750,None,-0.060631,-25000.0


# Finding 1: Which campaign had the highest ROI?

In [40]:
campaign_summary_df.sort_values('Campaign_Promo_ROI', ascending=False).head(1)

,Promo_Revenue_USD,Expected_Baseline_Revenue,Revenue_Uplift,Discount_Cost,Promo_Units_Sold,Expected_Baseline_Units,Campaign_Promo_ROI,Campaign_Volume_Uplift_percent,Campaign_Net_Promo_Gain
Campaign_Name,,,,,,,,,
Black Friday,22400000.0,15900000.0,6500000.0,3975000.0,204199,144946.375,163.522013,40.878997,2525000.0


# Finding 2: Which campaign had the lowest ROI — or negative ROI?

In [ ]:
campaign_summary_df.sort_values('Campaign_Promo_ROI', ascending=True).head(1)

# Finding 3: Which campaign drove the highest volume uplift %?

In [41]:
campaign_summary_df.sort_values('Campaign_Volume_Uplift_percent', ascending=False).head(1)

,Promo_Revenue_USD,Expected_Baseline_Revenue,Revenue_Uplift,Discount_Cost,Promo_Units_Sold,Expected_Baseline_Units,Campaign_Promo_ROI,Campaign_Volume_Uplift_percent,Campaign_Net_Promo_Gain
Campaign_Name,,,,,,,,,
Black Friday,22400000.0,15900000.0,6500000.0,3975000.0,204199,144946.375,163.522013,40.878997,2525000.0


# Finding 4: Total revenue uplift across all campaigns combined

In [42]:
campaign_summary_df['Revenue_Uplift'].sum()

7099999.999999996

In [43]:
sales_df.to_excel('Sales_Data_Clean.xlsx')

<ipython-input-43-1273c0e5fcc6>:1: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.2.9' currently installed).
  sales_df.to_excel('Sales_Data_Clean.xlsx')


In [47]:
promo_analysis_df.to_excel('Promo_details.xlsx')

<ipython-input-47-76fa8b93d46f>:1: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.2.9' currently installed).
  promo_analysis_df.to_excel('Promo_details.xlsx')


In [48]:
campaign_summary_df.to_excel('promo_campaign_summary.xlsx')

<ipython-input-48-19b55717fb5b>:1: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.2.9' currently installed).
  campaign_summary_df.to_excel('promo_campaign_summary.xlsx')
